# Long-form DuckDB build + gene-axis diagnostic

Diagnostic only — no scoring, no Noisy-OR, no confidence. Goal: build the long-form analytical table exactly per the Track D architecture spec, then empirically measure what a gene query can and cannot reach across the 14 canonical parquet tables in `cleaned_track_data/`.

Canonical keys: `model_id` (uppercase `ACH-...`), `ensg_id` (bare Ensembl, no version suffix).

In [ ]:
import re
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../../cleaned_track_data")
REF_DIR = Path("../../reference")

ALL_LAYERS = [
    "Cellosaurus", "depmap_profile", "fusions_gene_level", "fusions_scores",
    "geo_info", "hpa_desc_resolved", "metabolomics", "mirna_model_level",
    "mutations_collapsed", "mutations_scores", "mutations_variant_detail",
    "proteomics", "sample_info", "signatures_model_level",
]
assert len(ALL_LAYERS) == 14

ENSG_RE = re.compile(r"^ENSG\d+$")


## Step 1 — Build the long-form table

Schema (fixed, not modified): `(model_id VARCHAR, layer VARCHAR, feature_id VARCHAR, value DOUBLE)`.

Two normalisations only: `model_id` uppercased; `ensg_id` stripped of version suffix (`.str.split(".").str[0]`).

**5 of the 14 tables carry no measurement feature axis at all** — `Cellosaurus`, `depmap_profile`, `geo_info`, `hpa_desc_resolved`, `sample_info` are identity/dimension tables per the architecture doc (they carry alternative identifiers for the `cell_line` connector: rrids, ccle_names, geo_accessions, profile_ids — not a gene/protein/metabolite axis). They are declared as layers below and contribute **0 rows** to `long`. That absence is the point of Step 4, not something to patch.

For wide-matrix tables (`metabolomics`, `proteomics`, `signatures_model_level`), every non-key column is melted: `feature_id` = column name, `value` = that column's value. Non-numeric columns fail the numeric cast and are dropped automatically — that's why `ccle_id` in `metabolomics` disappears with no special-casing.

For tables already in long form with an explicit feature-axis column (`fusions_gene_level`, `fusions_scores`, `mutations_collapsed`, `mutations_scores`, `mutations_variant_detail`, `mirna_model_level`), `feature_id` is that column (`ensg_id` or `mirna_id`), and one representative numeric column is kept as `value`:

| table | feature_id | value | why |
|---|---|---|---|
| fusions_gene_level | ensg_id | fusion_count | plain count, always defined (`max_confidence` is categorical text: low/medium/high, not numeric) |
| fusions_scores | ensg_id | p_fusion | the scored probability |
| mutations_collapsed | ensg_id | variant_burden | aggregate gene-level scalar |
| mutations_scores | ensg_id | p_mutation | the scored probability |
| mutations_variant_detail | ensg_id | af | allele fraction (variant-level rows — many rows per gene, by design) |
| mirna_model_level | mirna_id | expression_value | the only measurement column present |

None of these choices affect the reach diagnostic in Steps 3–4: `COUNT(*)` / `COUNT(DISTINCT model_id)` only depend on whether a row exists for `(model_id, layer, feature_id)`, not on which numeric column was kept as `value`.

In [ ]:
def norm_model_id(s: pd.Series) -> pd.Series:
    return s.astype("string").str.upper()


def norm_ensg(s: pd.Series) -> pd.Series:
    return s.astype("string").str.split(".").str[0]


def wide_melt(df: pd.DataFrame, key_col: str, layer_name: str) -> pd.DataFrame:
    """Every non-key column becomes its own feature_id; values that fail a numeric cast drop out."""
    other_cols = [c for c in df.columns if c != key_col]
    m = df[[key_col] + other_cols].melt(id_vars=key_col, var_name="feature_id", value_name="value")
    m = m.rename(columns={key_col: "model_id"})
    m["value"] = pd.to_numeric(m["value"], errors="coerce")
    m = m.dropna(subset=["value"])
    m["model_id"] = norm_model_id(m["model_id"])
    m["layer"] = layer_name
    return m[["model_id", "layer", "feature_id", "value"]]


def long_from_feature_col(df: pd.DataFrame, key_col: str, feature_col: str, value_col: str, layer_name: str) -> pd.DataFrame:
    m = df[[key_col, feature_col, value_col]].copy()
    m = m.rename(columns={key_col: "model_id", feature_col: "feature_id", value_col: "value"})
    m["value"] = pd.to_numeric(m["value"], errors="coerce")
    m = m.dropna(subset=["value"])
    m["model_id"] = norm_model_id(m["model_id"])
    m["feature_id"] = norm_ensg(m["feature_id"].astype("string"))
    m["layer"] = layer_name
    return m[["model_id", "layer", "feature_id", "value"]]


In [ ]:
frames = []

# Identity / dimension tables carry no feature axis -> deliberately contribute 0 rows.
# (Cellosaurus, depmap_profile, geo_info, hpa_desc_resolved, sample_info)

metabolomics = pd.read_parquet(DATA_DIR / "metabolomics.parquet", engine="pyarrow").drop(columns=["ccle_id"], errors="ignore")
frames.append(wide_melt(metabolomics, "model_id", "metabolomics"))

proteomics = pd.read_parquet(DATA_DIR / "proteomics.parquet", engine="pyarrow")
frames.append(wide_melt(proteomics, "depmap_id", "proteomics"))

signatures = pd.read_parquet(DATA_DIR / "signatures_model_level.parquet", engine="pyarrow")
frames.append(wide_melt(signatures, "model_id", "signatures_model_level"))

fusions_gene_level = pd.read_parquet(DATA_DIR / "fusions_gene_level.parquet", engine="pyarrow")
frames.append(long_from_feature_col(fusions_gene_level, "model_id", "ensg_id", "fusion_count", "fusions_gene_level"))

fusions_scores = pd.read_parquet(DATA_DIR / "fusions_scores.parquet", engine="pyarrow")
frames.append(long_from_feature_col(fusions_scores, "model_id", "ensg_id", "p_fusion", "fusions_scores"))

mirna = pd.read_parquet(DATA_DIR / "mirna_model_level.parquet", engine="pyarrow")
frames.append(long_from_feature_col(mirna, "model_id", "mirna_id", "expression_value", "mirna_model_level"))

mutations_collapsed = pd.read_parquet(DATA_DIR / "mutations_collapsed.parquet", engine="pyarrow")
frames.append(long_from_feature_col(mutations_collapsed, "model_id", "ensg_id", "variant_burden", "mutations_collapsed"))

mutations_scores = pd.read_parquet(DATA_DIR / "mutations_scores.parquet", engine="pyarrow")
frames.append(long_from_feature_col(mutations_scores, "model_id", "ensg_id", "p_mutation", "mutations_scores"))

mutations_variant_detail = pd.read_parquet(DATA_DIR / "mutations_variant_detail.parquet", engine="pyarrow")
frames.append(long_from_feature_col(mutations_variant_detail, "model_id", "ensg_id", "af", "mutations_variant_detail"))

long_df = pd.concat(frames, ignore_index=True)
len(long_df)


In [ ]:
con = duckdb.connect()
con.execute("""
    CREATE TABLE long (
        model_id VARCHAR,
        layer VARCHAR,
        feature_id VARCHAR,
        value DOUBLE
    )
""")
con.execute("INSERT INTO long SELECT model_id, layer, feature_id, value FROM long_df")

all_layers_df = pd.DataFrame({"layer": ALL_LAYERS})
con.register("all_layers_df", all_layers_df)

con.sql("SELECT COUNT(*) AS total_rows FROM long").show()


In [ ]:
rows_per_layer = con.sql("""
    SELECT al.layer, COUNT(l.model_id) AS n_rows
    FROM all_layers_df al
    LEFT JOIN long l ON l.layer = al.layer
    GROUP BY al.layer
    ORDER BY al.layer
""").df()
rows_per_layer


## Step 2 — What does `feature_id` actually contain, per layer?

No interpretation here — just the printed inventory the task asks for.

In [ ]:
step2_rows = []
for layer in ALL_LAYERS:
    distinct_df = con.sql(f"""
        SELECT DISTINCT feature_id FROM long WHERE layer = \'{layer}\'
    """).df()
    n_distinct = len(distinct_df)
    examples = distinct_df["feature_id"].head(5).tolist()
    if n_distinct == 0:
        ensg_frac = None
    else:
        ensg_frac = distinct_df["feature_id"].str.match(ENSG_RE).mean()
    step2_rows.append({
        "layer": layer,
        "n_distinct_feature_id": n_distinct,
        "ensg_pattern_fraction": ensg_frac,
        "example_feature_ids": examples,
    })

step2 = pd.DataFrame(step2_rows)
step2


## Step 3 — Gene query reach

**Note on the task's gene list:** the task specifies KRAS as `ENSG00000133024`. That id does not resolve to anything in `reference/gene_lookup.parquet` (the project's own canonical symbol → ENSG table) — it is not a valid protein-coding gene id in this project at all. KRAS's actual id is `ENSG00000133703`. Querying the id as literally given would show 0 rows in all 14 layers for the wrong reason (invalid id, not "gene lacks a measurement"), which would confound exactly the distinction Step 4 is meant to draw. The corrected id is used below and verified against `gene_lookup` first.

In [ ]:
gene_lookup = pd.read_parquet(REF_DIR / "gene_lookup.parquet", engine="pyarrow")

# Verify all three anchor genes exist in gene_lookup before querying
anchor_genes = {
    "BRAF": "ENSG00000157764",
    "KRAS": "ENSG00000133703",
    "EGFR": "ENSG00000146648",
}

for name, ensg in anchor_genes.items():
    hit = gene_lookup[gene_lookup.ensg_id == ensg]
    status = "FOUND" if len(hit) > 0 else "MISSING"
    print(f"{name}: {ensg} \u2014 {status}")


In [ ]:
def gene_reach(ensg_id: str) -> pd.DataFrame:
    return con.sql(f"""
        SELECT al.layer,
               COUNT(l.model_id) AS n_rows,
               COUNT(DISTINCT l.model_id) AS n_cell_lines
        FROM all_layers_df al
        LEFT JOIN long l
          ON l.layer = al.layer AND l.feature_id = \'{ensg_id}\'
        GROUP BY al.layer
        ORDER BY al.layer
    """).df()

reach = {name: gene_reach(ensg) for name, ensg in anchor_genes.items()}

for name, df in reach.items():
    print(f"=== {name} ({anchor_genes[name]}) ===")
    print(df.to_string(index=False))
    print()


## Step 4 — Report only: reach vs. axis, per gene

No fixes, no redesign — just the count and the classification the task asks for.

In [ ]:
gene_axis_layers = set(step2.loc[step2["ensg_pattern_fraction"] == 1.0, "layer"])
no_axis_layers = set(ALL_LAYERS) - gene_axis_layers

for name, df in reach.items():
    n_hit = int((df["n_rows"] > 0).sum())
    n_zero = int((df["n_rows"] == 0).sum())
    zero_layers = df.loc[df["n_rows"] == 0, "layer"].tolist()

    print(f"=== {name} ({anchor_genes[name]}) ===")
    print(f"{n_hit}/14 layers returned rows, {n_zero}/14 returned zero.")
    for layer in zero_layers:
        reason = (
            "gene has no measurement in a layer that DOES have a gene axis"
            if layer in gene_axis_layers
            else "layer has no gene axis at all"
        )
        print(f"  - {layer}: {reason}")
    print()

print("Layers with a gene (ENSG) axis:", sorted(gene_axis_layers))
print("Layers with no gene axis at all:", sorted(no_axis_layers))


## Step 1 (revised) — Fix the UniProt→ENSG join and rebuild

Proteomics feature IDs are UniProt accessions (lower-cased); the original build left them
unmapped so `proteomics` showed `ensg_fraction = 0.0`.
Fix: join `proteomics` long rows to `gene_lookup.uniprot_ids` → `ensg_id` before inserting
into `long2`.  All other layers are identical to Step 1 above.

In [ ]:
import re
from pathlib import Path

import duckdb
import pandas as pd

DATA_DIR = Path("../../cleaned_track_data")
REF_DIR  = Path("../../reference")
ENSG_RE  = re.compile(r"^ENSG\d+$")

ALL_LAYERS = [
    "Cellosaurus", "depmap_profile", "fusions_gene_level", "fusions_scores",
    "geo_info", "hpa_desc_resolved", "metabolomics", "mirna_model_level",
    "mutations_collapsed", "mutations_scores", "mutations_variant_detail",
    "proteomics", "sample_info", "signatures_model_level",
]

gene_lookup = pd.read_parquet(REF_DIR / "gene_lookup.parquet", engine="pyarrow")

# UniProt -> ENSG lookup; strip+lowercase to match proteomics column names
uniprot_to_ensg = (
    gene_lookup[["ensg_id", "uniprot_ids"]]
    .dropna(subset=["uniprot_ids"])
    .rename(columns={"uniprot_ids": "uniprot_id"})
    .assign(uniprot_id=lambda x: x["uniprot_id"].str.strip().str.lower())
    .drop_duplicates(subset=["uniprot_id"])
)

print(f"UniProt->ENSG lookup rows: {len(uniprot_to_ensg)}")
print(uniprot_to_ensg.head(3))


In [ ]:
frames2 = []

# Identity / dimension tables carry no feature axis -> deliberately 0 rows.

metabolomics = pd.read_parquet(DATA_DIR / "metabolomics.parquet", engine="pyarrow").drop(columns=["ccle_id"], errors="ignore")
frames2.append(wide_melt(metabolomics, "model_id", "metabolomics"))

# Proteomics: map UniProt IDs -> ENSG via gene_lookup
proteomics_raw = pd.read_parquet(DATA_DIR / "proteomics.parquet", engine="pyarrow")
prot_melted = wide_melt(proteomics_raw, "depmap_id", "proteomics")
prot_mapped = prot_melted.merge(
    uniprot_to_ensg, left_on="feature_id", right_on="uniprot_id", how="inner"
)
prot_mapped = prot_mapped.assign(feature_id=prot_mapped["ensg_id"])[
    ["model_id", "layer", "feature_id", "value"]
]
print(f"proteomics before map: {len(prot_melted)}, after map: {len(prot_mapped)}")
frames2.append(prot_mapped)

signatures = pd.read_parquet(DATA_DIR / "signatures_model_level.parquet", engine="pyarrow")
frames2.append(wide_melt(signatures, "model_id", "signatures_model_level"))

fusions_gene_level = pd.read_parquet(DATA_DIR / "fusions_gene_level.parquet", engine="pyarrow")
frames2.append(long_from_feature_col(fusions_gene_level, "model_id", "ensg_id", "fusion_count", "fusions_gene_level"))

fusions_scores = pd.read_parquet(DATA_DIR / "fusions_scores.parquet", engine="pyarrow")
frames2.append(long_from_feature_col(fusions_scores, "model_id", "ensg_id", "p_fusion", "fusions_scores"))

mirna = pd.read_parquet(DATA_DIR / "mirna_model_level.parquet", engine="pyarrow")
frames2.append(long_from_feature_col(mirna, "model_id", "mirna_id", "expression_value", "mirna_model_level"))

mutations_collapsed = pd.read_parquet(DATA_DIR / "mutations_collapsed.parquet", engine="pyarrow")
frames2.append(long_from_feature_col(mutations_collapsed, "model_id", "ensg_id", "variant_burden", "mutations_collapsed"))

mutations_scores = pd.read_parquet(DATA_DIR / "mutations_scores.parquet", engine="pyarrow")
frames2.append(long_from_feature_col(mutations_scores, "model_id", "ensg_id", "p_mutation", "mutations_scores"))

mutations_variant_detail = pd.read_parquet(DATA_DIR / "mutations_variant_detail.parquet", engine="pyarrow")
frames2.append(long_from_feature_col(mutations_variant_detail, "model_id", "ensg_id", "af", "mutations_variant_detail"))

long_df2 = pd.concat(frames2, ignore_index=True)
print(f"Total rows in long2: {len(long_df2)}")

con2 = duckdb.connect()
con2.execute("""
    CREATE TABLE long (
        model_id VARCHAR, layer VARCHAR, feature_id VARCHAR, value DOUBLE
    )
""")
con2.execute("INSERT INTO long SELECT model_id, layer, feature_id, value FROM long_df2")

all_layers_df2 = pd.DataFrame({"layer": ALL_LAYERS})
con2.register("all_layers_df2", all_layers_df2)
con2.sql("SELECT COUNT(*) AS total_rows FROM long").show()


## Step 2 (revised) — Feature axis inventory

Same logic as before but run against `long2` / `con2`, which now has ENSG-mapped proteomics.

In [ ]:
step2_rows2 = []
for layer in ALL_LAYERS:
    distinct_df = con2.sql(
        f"SELECT DISTINCT feature_id FROM long WHERE layer = '{layer}'"
    ).df()
    n_distinct = len(distinct_df)
    examples = distinct_df["feature_id"].head(5).tolist()
    ensg_frac = distinct_df["feature_id"].str.match(ENSG_RE).mean() if n_distinct > 0 else None
    step2_rows2.append({
        "layer": layer,
        "n_distinct_feature_id": n_distinct,
        "ensg_pattern_fraction": ensg_frac,
        "example_feature_ids": examples,
    })

step2_v2 = pd.DataFrame(step2_rows2)
step2_v2


## Step 3 (revised) — Gene query reach

Same anchor genes, now run against `con2` so the proteomics layer is ENSG-keyed.

In [ ]:
anchor_genes = {
    "BRAF": "ENSG00000157764",
    "KRAS": "ENSG00000133703",
    "EGFR": "ENSG00000146648",
}

for name, ensg in anchor_genes.items():
    hit = gene_lookup[gene_lookup.ensg_id == ensg]
    print(f"{name}: {ensg} \u2014 {'FOUND' if len(hit) > 0 else 'MISSING'}")

def gene_reach2(ensg_id: str) -> pd.DataFrame:
    return con2.sql(
        f"""
        SELECT al.layer,
               COUNT(l.model_id) AS n_rows,
               COUNT(DISTINCT l.model_id) AS n_cell_lines
        FROM all_layers_df2 al
        LEFT JOIN long l
          ON l.layer = al.layer AND l.feature_id = '{ensg_id}'
        GROUP BY al.layer
        ORDER BY al.layer
        """
    ).df()

reach2 = {name: gene_reach2(ensg) for name, ensg in anchor_genes.items()}
for name, df in reach2.items():
    print(f"=== {name} ({anchor_genes[name]}) ===")
    print(df.to_string(index=False))
    print()


## Step 4 (revised) — Identity spine argument

In [ ]:
gene_axis_layers2 = set(step2_v2.loc[step2_v2["ensg_pattern_fraction"] == 1.0, "layer"])
no_axis_layers2   = set(ALL_LAYERS) - gene_axis_layers2

for name, df in reach2.items():
    n_hit  = int((df["n_rows"] > 0).sum())
    n_zero = int((df["n_rows"] == 0).sum())
    zero_layers = df.loc[df["n_rows"] == 0, "layer"].tolist()
    print(f"=== {name} ({anchor_genes[name]}) ===")
    print(f"{n_hit}/14 layers returned rows, {n_zero}/14 returned zero.")
    for layer in zero_layers:
        reason = (
            "gene has no measurement in a layer that DOES have a gene axis"
            if layer in gene_axis_layers2
            else "layer has no gene axis at all"
        )
        print(f"  - {layer}: {reason}")
    print()

print("Layers with a gene (ENSG) axis:", sorted(gene_axis_layers2))
print("Layers with no gene axis at all:", sorted(no_axis_layers2))
print("""
=== IDENTITY SPINE ARGUMENT ===
- model_id alone is sufficient to JOIN all 14 tables (cell-line axis).
- ensg_id is required to QUERY by gene (gene axis).
- Layers with no gene axis (metabolomics, miRNA, signatures, identity tables)
  are unreachable by a gene query — correct by design; they are cell-line
  modifiers or alternative-identifier registries, not per-gene evidence.
- Track A (identity spine) provides model_id only — correct.
- ensg_id must be carried by evidence tables themselves.
""")


## Step 5 — Continuous vs discrete checkpoint: `depmap_expr`

Load expression and report shape, column names, and index format before writing the
comparison step.  The comparison code depends on whether ENSG IDs are lowercased columns
and whether `model_id` is the index or a column — **stop here and report back**.

In [ ]:
# depmap_expr_clean is in data/parquet/data_clean/, not cleaned_track_data/
# Index values are profile IDs (pr-...), not ACH- model_ids -- join via depmap_profile needed
EXPR_PATH = Path("../../data/parquet/data_clean/depmap_expr_clean.parquet")
depmap_expr = pd.read_parquet(EXPR_PATH, engine="pyarrow")
print("Shape:", depmap_expr.shape)
print("First 5 column names:", depmap_expr.columns[:5].tolist())
print("Index type:", type(depmap_expr.index).__name__)
print("First 3 index values:", depmap_expr.index[:3].tolist())
print("model_id is column?", "model_id" in depmap_expr.columns)
print("Columns are lowercase ENSG?", depmap_expr.columns[0].startswith("ensg"))


In [ ]:

EXPR_PATH = Path("../../data/parquet/data_clean/depmap_expr_clean.parquet")
depmap_expr = pd.read_parquet(EXPR_PATH)
print(depmap_expr.shape)
print(depmap_expr.columns[:5].tolist())
print(depmap_expr.index[:3].tolist())
print(type(depmap_expr.index[0]))

In [ ]:
profiles = pd.read_parquet(REF_DIR / "depmap_profiles.parquet")
print(profiles.columns.tolist())
print(profiles.head(3))

In [ ]:
# --- Setup ---
# Map profile_id → model_id
profiles = profiles[["profileid", "modelid"]].copy()
profiles["modelid"] = profiles["modelid"].str.upper()

# Anchor genes — uppercase the column lookup
anchor_genes = {
    "BRAF": "ensg00000157764",  # lowercase to match depmap_expr columns
    "KRAS": "ensg00000133703",
    "EGFR": "ensg00000146648",
}

# Pull expression for anchor genes, join to model_id
expr_subset = depmap_expr[list(anchor_genes.values())].copy()
expr_subset.index.name = "profileid"
expr_subset = expr_subset.reset_index()
expr_subset = expr_subset.merge(profiles, on="profileid", how="inner")
expr_subset = expr_subset.rename(columns={"modelid": "model_id"})
expr_subset = expr_subset.drop(columns=["profileid"])

print(f"Rows after profile join: {len(expr_subset)}")
print(expr_subset.head(3))

In [ ]:
depmap_profiles = pd.read_parquet(REF_DIR / "depmap_profiles.parquet", engine="pyarrow")
# filter to RNA profiles, then upper-case modelid to match ACH- convention
rna_profiles = depmap_profiles[depmap_profiles["datatype"] == "rna"][["profileid", "modelid"]]
rna_profiles["model_id"] = rna_profiles["modelid"].str.upper()

## Deliverable

This notebook is diagnostic only. No scoring, no Noisy-OR, no confidence weighting, and no fix is proposed for the zero-reach layers above — that is a separate, later decision.

In [ ]:
import numpy as np

# Reshape to long form for comparison
expr_long = expr_subset.melt(
    id_vars="model_id",
    var_name="ensg_id_lower",
    value_name="expression"
)
expr_long["ensg_id"] = expr_long["ensg_id_lower"].str.upper()
expr_long = expr_long.drop(columns=["ensg_id_lower"])

# --- Branch C: Continuous ---
# Normalise to [0,1] per gene using min-max across all cell lines
for gene in expr_long["ensg_id"].unique():
    mask = expr_long["ensg_id"] == gene
    vals = expr_long.loc[mask, "expression"]
    mn, mx = vals.min(), vals.max()
    expr_long.loc[mask, "score_continuous"] = (vals - mn) / (mx - mn)

# --- Branch T: Discrete (HPA paradigm) ---
# Threshold per gene: present if expression > 1 log2-TPM (standard HPA cutoff)
# Confidence = fraction of lines where gene is present (simple cross-line signal)
HPA_THRESHOLD = 1.0  # log2-TPM
expr_long["score_discrete"] = (expr_long["expression"] > HPA_THRESHOLD).astype(float)

# --- Compare distributions ---
print("=== Continuous score distribution per gene ===")
print(expr_long.groupby("ensg_id")["score_continuous"].describe().round(3))

print("\n=== Discrete score distribution per gene ===")
print(expr_long.groupby("ensg_id")["score_discrete"].describe().round(3))

# --- Find where they disagree most ---
expr_long["disagreement"] = abs(
    expr_long["score_continuous"] - expr_long["score_discrete"]
)
print("\n=== Top 10 lines where continuous vs discrete disagree most (BRAF) ===")
braf = expr_long[expr_long["ensg_id"] == "ENSG00000157764"].sort_values(
    "disagreement", ascending=False
)
print(braf[["model_id", "expression", "score_continuous",
            "score_discrete", "disagreement"]].head(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
genes = {
    "BRAF": "ENSG00000157764",
    "KRAS": "ENSG00000133703",
    "EGFR": "ENSG00000146648",
}

for ax, (name, ensg) in zip(axes, genes.items()):
    subset = expr_long[expr_long["ensg_id"] == ensg]
    ax.scatter(
        subset["score_continuous"],
        subset["score_discrete"],
        alpha=0.3, s=10
    )
    ax.axvline(x=0.5, color="red", linestyle="--", linewidth=0.8)
    ax.axhline(y=0.5, color="red", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Continuous score")
    ax.set_ylabel("Discrete score")
    ax.set_title(f"{name}")

plt.suptitle("Continuous vs Discrete scoring — disagreement zones", y=1.02)
plt.tight_layout()
plt.savefig("continuous_vs_discrete.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: continuous_vs_discrete.png")